# Slider 강화학습 (알파제로 방식, Colab GPU)

신경망이 **수마다의 확률(정책)** 과 **이길 확률(가치)** 을 내고, MCTS 로 자가대전한 판의 기록과 승패로 스스로 배웁니다.
규칙은 Rust 엔진과 똑같다고 확인한 JAX 구현(`gpu/slider_jax.py`), 학습 코드는 `gpu/alphazero.py` 입니다.

- 런타임: **GPU (T4 이상)**. 런타임 > 런타임 유형 변경.
- 반복 1번 = 판 `GAMES` 개를 동시에 MCTS 로 끝까지 둠 -> 그 기록으로 신경망 학습. 판 수를 늘릴수록 데이터가 많아집니다.
- 승패: 이기면 +1, 지면 -1, 무승부(같은 국면 3번, 또는 `MAX_LEN` 수까지 안 끝남) 0.
- 몇 번마다 **전술 상대**(바로 이기는 수는 두고, 상대에게 바로 이기는 수를 주지 않는 무작위)와 **무작위 상대**에게 대국시켜 점수를 봅니다.
- 반복마다 Google Drive `slider_results/alphazero/` 에 저장됩니다. 끊기면 처음부터 다시 실행하면 이어서 학습합니다.
- 학습 셀은 언제든 멈춰도 됩니다 (마지막 저장까지 남음).

In [ ]:
# ===== 설정 =====
BRANCH = "rust-port"
GAMES = 512            # 반복마다 동시에 두는 자가대전 판 수 (많을수록 데이터가 많음. 메모리가 모자라면 줄이세요)
SIMULATIONS = 32       # 수마다 MCTS 시뮬레이션 수 (많을수록 수가 정확하지만 느림)
MAX_LEN = 200          # 이 수까지 안 끝나면 무승부
HIDDEN = 256           # 신경망 은닉 크기
LR = 1e-3
TRAIN_STEPS = 200      # 반복마다 학습 횟수
MINIBATCH = 1024
REPLAY = 8             # 최근 몇 반복의 기록으로 학습할지
ITERATIONS = 200       # 총 반복 수 (중간에 멈춰도 됨)
EVAL_EVERY = 5
EVAL_GAMES = 256
SEED = 0
SAVE_DIR = "/content/drive/MyDrive/slider_results/alphazero"

In [ ]:
import os, sys, json, time, pathlib, subprocess
from google.colab import drive
drive.mount("/content/drive")
os.makedirs(SAVE_DIR, exist_ok=True)
if not os.path.exists("/content/Slider"):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", BRANCH, "https://github.com/Perdume/Slider.git", "/content/Slider"], check=True)
else:
    subprocess.run(["git", "-C", "/content/Slider", "pull", "-q", "--ff-only"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "mctx", "optax"], check=True)
sys.path.insert(0, "/content/Slider/gpu")
import numpy as np
import jax, jax.numpy as jnp
import alphazero as az
print("device:", jax.devices())

## 이어서 학습할 저장 파일 불러오기

In [ ]:
CKPT = os.path.join(SAVE_DIR, f"ckpt_h{HIDDEN}.pkl")
opt = az.make_optimizer(LR)
if os.path.exists(CKPT):
    ck = az.load(CKPT)
    params = jax.tree_util.tree_map(jnp.asarray, ck["params"])
    opt_state = jax.tree_util.tree_map(jnp.asarray, ck["opt_state"])
    start_it, history = ck["iteration"], ck["history"]
    print(f"이어서 학습: 반복 {start_it} 부터")
else:
    params = az.init_params(jax.random.PRNGKey(SEED), hidden=HIDDEN)
    opt_state = opt.init(params)
    start_it, history = 0, []
    print("처음부터 학습")
replay = []   # (입력 uint8, 정책 float16, 가치 float32)

## 학습 (멈춰도 됨)

In [ ]:
rng = np.random.default_rng(SEED + start_it)
key = jax.random.PRNGKey(SEED + 1000 + start_it)
for it in range(start_it, ITERATIONS):
    t0 = time.time()
    key, k_play, k_eval = jax.random.split(key, 3)
    obs, pi, active, winner = az.self_play(params, k_play, batch=GAMES, num_simulations=SIMULATIONS, max_len=MAX_LEN)
    obs, pi, active, winner = (np.asarray(x) for x in (obs, pi, active, winner))
    z = az.value_targets(active, winner)
    replay.append((obs[active].astype(np.uint8), pi[active].astype(np.float16), z[active]))
    replay = replay[-REPLAY:]
    t_play = time.time() - t0

    X = np.concatenate([r[0] for r in replay]); P = np.concatenate([r[1] for r in replay]); Z = np.concatenate([r[2] for r in replay])
    losses = []
    for _ in range(TRAIN_STEPS):
        idx = rng.integers(0, len(Z), size=min(MINIBATCH, len(Z)))
        params, opt_state, loss, (pl, vl) = az.train_step(params, opt_state, jnp.asarray(X[idx], jnp.float32),
                                                          jnp.asarray(P[idx], jnp.float32), jnp.asarray(Z[idx]), opt)
        losses.append((float(pl), float(vl)))
    pl, vl = np.mean(losses, axis=0)

    lengths = active.sum(0)
    row = {"iteration": it + 1, "games": GAMES, "samples": int(active.sum()), "avg_len": float(lengths.mean()),
           "first_win": float((winner == 0).mean()), "second_win": float((winner == 1).mean()), "draw": float((winner < 0).mean()),
           "policy_loss": float(pl), "value_loss": float(vl), "sec_play": t_play, "sec_total": time.time() - t0}
    if (it + 1) % EVAL_EVERY == 0:
        for opp in ("greedy", "random"):
            w, d, l = (int(x) for x in az.evaluate(params, k_eval, batch=EVAL_GAMES, num_simulations=SIMULATIONS,
                                                   max_len=MAX_LEN, opponent=opp))
            row[f"vs_{opp}"] = [w, d, l]
            row[f"vs_{opp}_score"] = (w + 0.5 * d) / EVAL_GAMES
    history.append(row)
    msg = (f"[{it + 1}] {row['sec_total']:.0f}s | 평균 {row['avg_len']:.0f}수, 선승 {row['first_win']:.0%} 후승 {row['second_win']:.0%} "
           f"무 {row['draw']:.0%} | 손실 정책 {pl:.3f} 가치 {vl:.3f}")
    if "vs_greedy" in row:
        msg += f" | 전술 상대 {row['vs_greedy']} ({row['vs_greedy_score']:.2f}), 무작위 {row['vs_random']} ({row['vs_random_score']:.2f})"
    print(msg, flush=True)
    az.save(CKPT, params, opt_state, it + 1, history)
    with open(os.path.join(SAVE_DIR, f"history_h{HIDDEN}.json"), "w") as f:
        json.dump(history, f, ensure_ascii=False, indent=1)

## 그래프

In [ ]:
import matplotlib.pyplot as plt
its = [h["iteration"] for h in history]
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
ax[0].plot(its, [h["policy_loss"] for h in history], label="정책 손실"); ax[0].plot(its, [h["value_loss"] for h in history], label="가치 손실"); ax[0].legend(); ax[0].set_title("loss")
ax[1].plot(its, [h["avg_len"] for h in history]); ax[1].set_title("avg game length")
ev = [h for h in history if "vs_greedy_score" in h]
ax[2].plot([h["iteration"] for h in ev], [h["vs_greedy_score"] for h in ev], label="vs greedy")
ax[2].plot([h["iteration"] for h in ev], [h["vs_random_score"] for h in ev], label="vs random"); ax[2].set_ylim(0, 1); ax[2].legend(); ax[2].set_title("eval score")
plt.show()